# FinReason — Setup & Data Exploration
**Course:** EGN 6217 — Applied Deep Learning | Spring 2026 | **Author:** Om

## 1. Environment Check

In [ ]:
import torch, platform
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda if torch.cuda.is_available() else "N/A"}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
for p in ['transformers','trl','peft','bitsandbytes','accelerate','datasets','streamlit']:
    try: m=__import__(p); print(f'  ✓ {p}: {getattr(m,"__version__","ok")}')
    except: print(f'  ✗ {p}: MISSING')

## 2. Load FinQA

In [ ]:
from datasets import load_dataset
dataset = load_dataset('wandb/finqa-data-processed')
for s in dataset: print(f'  {s}: {len(dataset[s])}')
print(f'Columns: {dataset["train"].column_names}')

## 3. Data Structure

In [ ]:
ex = dataset['train'][0]
for k in ex:
    v = str(ex[k])
    print(f'  {k}: {v[:200]}...' if len(v)>200 else f'  {k}: {v}')

## 4. Extraction Functions

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from shared_utils import extract_qa, extract_context
q, a = extract_qa(dataset['train'][0])
ctx = extract_context(dataset['train'][0])
print(f'Q: {q}\nA: {a}\nContext: {len(ctx)} chars')

## 5. Answer Distribution

In [ ]:
import matplotlib.pyplot as plt, re, numpy as np
from collections import Counter
train = dataset['train']
types = Counter(); ctx_lens = []; ans_lens = []
for ex in train:
    _,a = extract_qa(ex); ctx = extract_context(ex)
    ans_lens.append(len(a.strip())); ctx_lens.append(len(ctx))
    try: float(re.sub(r'[,$%]','',a.strip())); types['numeric']+=1
    except: types['yes/no' if a.strip().lower() in ('yes','no') else 'text']+=1
for t,c in types.most_common(): print(f'  {t}: {c} ({100*c/len(train):.1f}%)')
fig,axes=plt.subplots(1,3,figsize=(14,4))
axes[0].bar(types.keys(),types.values(),color=['#3b82f6','#10b981','#f59e0b']); axes[0].set_title('Answer Types')
axes[1].hist(ctx_lens,bins=50,color='#3b82f6'); axes[1].set_title('Context Length')
axes[2].hist(ans_lens,bins=30,color='#10b981'); axes[2].set_title('Answer Length')
plt.tight_layout(); os.makedirs('results',exist_ok=True); plt.savefig('results/data_exploration.png',dpi=150); plt.show()

## 6. Reward Function Test

In [ ]:
from shared_utils import extract_final_answer, check_answer, reward_function
tests = [('42.5','42.5',True),('$1,452.4','1452.4',True),('1.45B','1450000000',True),
    ('(3.5)','-3.5',True),('RM 12,825','12825',True),('50','42',False),
    ('<think>306.2</think>\n306.2','306.2',True),('Answer: 306.2','306.2',True)]
passed=0
for o,g,e in tests:
    ok=check_answer(extract_final_answer(o),g)==e; passed+=ok
    print(f'  {"✓" if ok else "✗"} {o[:40]:40s} vs {g}')
print(f'\n  {passed}/{len(tests)} passed')

## 7. Model Loading (4-bit)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL='Qwen/Qwen2.5-1.5B-Instruct'
tok=AutoTokenizer.from_pretrained(MODEL,)
mdl=AutoModelForCausalLM.from_pretrained(MODEL,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True),
    device_map='auto',)
print(f'VRAM: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')

## 8. Quick Zero-Shot Test

In [ ]:
from shared_utils import format_prompt
test_data=dataset['test'] if 'test' in dataset else dataset['validation']
for i in range(5):
    q,gt=extract_qa(test_data[i]); ctx=extract_context(test_data[i])
    ids=tok(format_prompt(ctx,q),return_tensors='pt',truncation=True,max_length=1024).to(mdl.device)
    with torch.no_grad(): out=mdl.generate(**ids,max_new_tokens=128,do_sample=False,pad_token_id=tok.eos_token_id)
    pred=tok.decode(out[0][ids['input_ids'].shape[1]:],skip_special_tokens=True).strip()
    final=extract_final_answer(pred); ok=check_answer(final,gt)
    print(f'  {"✓" if ok else "✗"} GT={gt} Pred={final}')
del mdl; torch.cuda.empty_cache()

## ✓ Setup Complete
- Dataset loaded ✓
- Reward function tested ✓
- Model loads in 4-bit ✓
- Zero-shot works ✓

**Next:** Run the training pipeline (SFT → GRPO)